# speed + uncertainty prototype (IO-VNBD)



## Scope and honest limits
- **Model input:** the six S-M phone accelerometer/gyroscope channels only.
- **Training label:** V-M ECU indicated speed, offline only. Phone GPS speed is deliberately not used; its comparison to ECU speed was inconsistent.
- **Sampling:** IO-VNBD phone data is about 10 Hz, so this uses a 26-sample (about 2.6 s) window. It does not upsample to the 100 Hz used in the referenced project.
- **Prediction:** speed at the end of the trailing window, not future speed.
- **Not yet a 2-D position solution:** S-M/V-M coordinates differ by roughly 9–10 km in the 50%/75% samples. Do not train position targets or claim the 3.5 m objective from this notebook.
- The 75% block was already inspected. It is reported once as a diagnostic only—do not tune features, thresholds, or architecture against it. One Driver B trip cannot establish cross-drive generalization.


In [1]:
# 1. Imports and bounded HTTP range access
import csv
import io
import urllib.request
import copy
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(7)
torch.manual_seed(7)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch:", torch.__version__, "| CPU prototype")

S_URL = (
    "https://media.githubusercontent.com/media/onyekpeu/IO-VNBD/master/"
    "Synchronised%20V%20abd%20S%20datasets/Categorised%20IOVNB%20Dataset/"
    "M%20(Driver%20B)/S-M.csv"
)
V_URL = (
    "https://media.githubusercontent.com/media/onyekpeu/IO-VNBD/master/"
    "Synchronised%20V%20abd%20S%20datasets/Categorised%20IOVNB%20Dataset/"
    "M%20(Driver%20B)/V-M.csv"
)
S_FILE_SIZE = 19_798_721
V_FILE_SIZE = 22_507_823

# These S/V ranges were previously checked and aligned; no full CSV is fetched.
RANGE_SPECS = {
    "prefix":     ((0, 1_048_575),          (0, 1_048_575)),
    "25% range":  ((4_949_680, 5_998_255),  (5_626_955, 6_675_530)),
    "50% range":  ((9_899_360, 10_947_935), (1_579_772, 3_152_635)),
    "75% range":  ((14_849_040, 15_897_615),(7_244_402, 8_817_265)),
}
TOTAL_BYTES_READ = 0

def fetch_range(url, start, end, file_size):
    """Fetch exactly one byte range; fail before reading if Range is ignored."""
    global TOTAL_BYTES_READ
    if not (0 <= start <= end < file_size):
        raise ValueError(f"Invalid range {start}-{end} for file size {file_size}")
    req = urllib.request.Request(
        url,
        headers={
            "Range": f"bytes={start}-{end}",
            "Accept-Encoding": "identity",
            "User-Agent": "Mozilla/5.0",
        },
    )
    with urllib.request.urlopen(req, timeout=60) as response:
        if response.status != 206:
            raise RuntimeError(
                f"Expected HTTP 206 for byte range, got {response.status}; stopped before reading body."
            )
        content_range = response.headers.get("Content-Range", "")
        if not content_range.startswith(f"bytes {start}-{end}/"):
            raise RuntimeError(f"Unexpected Content-Range: {content_range!r}")
        data = response.read()
    expected = end - start + 1
    if len(data) != expected:
        raise RuntimeError(f"Expected {expected} bytes, received {len(data)}")
    TOTAL_BYTES_READ += len(data)
    return data

def complete_text(data):
    text = data.decode("utf-8-sig", errors="replace")
    if not data.endswith(b"\n"):
        last_newline = text.rfind("\n")
        if last_newline < 0:
            return ""
        text = text[:last_newline + 1]
    return text

def clean_rows(rows):
    out = []
    for row in rows:
        cleaned = {}
        for key, value in row.items():
            if key is None:
                continue
            key = key.strip()
            cleaned[key] = value.strip() if isinstance(value, str) else value
        if any(v not in (None, "") for v in cleaned.values()):
            out.append(cleaned)
    return out

def parse_csv_range(data, start, fieldnames=None):
    text = complete_text(data)
    if not text:
        return [], fieldnames
    if start == 0:
        reader = csv.DictReader(io.StringIO(text))
        if not reader.fieldnames:
            raise RuntimeError("CSV header missing from prefix range")
        fieldnames = [name.strip() for name in reader.fieldnames]
        reader.fieldnames = fieldnames
        return clean_rows(reader), fieldnames
    if not fieldnames:
        raise ValueError("Nonzero range requires the prefix header")
    # A nonzero byte range can start mid-record; discard that partial first line.
    lines = text.splitlines()
    if lines:
        lines = lines[1:]
    reader = csv.DictReader(io.StringIO("\n".join(lines)), fieldnames=fieldnames)
    return clean_rows(reader), fieldnames

PyTorch: 2.14.1+cpu | CPU prototype


In [2]:
# 2. Read the selected ranges and identify the six phone channels
BLOCKS = {}
S_HEADERS = None
V_HEADERS = None
for block_name, (s_range, v_range) in RANGE_SPECS.items():
    s_bytes = fetch_range(S_URL, *s_range, S_FILE_SIZE)
    s_rows, S_HEADERS = parse_csv_range(s_bytes, s_range[0], S_HEADERS)
    v_bytes = fetch_range(V_URL, *v_range, V_FILE_SIZE)
    v_rows, V_HEADERS = parse_csv_range(v_bytes, v_range[0], V_HEADERS)
    BLOCKS[block_name] = {"S": s_rows, "V": v_rows}
    print(f"{block_name}: S={len(s_rows):,} rows, V={len(v_rows):,} rows")
print(f"Transferred {TOTAL_BYTES_READ:,} bytes total (bounded ranges only)")

def one_channel(prefix):
    hits = [name for name in S_HEADERS if name.upper().startswith(prefix.upper())]
    if len(hits) != 1:
        raise ValueError(f"Expected one channel starting {prefix!r}; found {hits}")
    return hits[0]

PHONE_CHANNELS = [
    one_channel("ACCELEROMETER X "),
    one_channel("ACCELEROMETER Y "),
    one_channel("ACCELEROMETER Z "),
    one_channel("GYROSCOPE YAW "),
    one_channel("GYROSCOPE PITCH "),
    one_channel("GYROSCOPE ROLL "),
]
S_TIME = "TIME SINCE START (ms)"
V_TIME = "Time Since Start of Day (seconds)"
V_ECU_SPEED = "Indicated Vehicle Speed (km/hr)"
PHONE_TO_VM_OFFSET_SECONDS = 29_604.173

for col in [S_TIME, *PHONE_CHANNELS]:
    if col not in S_HEADERS:
        raise KeyError(f"Missing S-M field: {col}")
for col in [V_TIME, V_ECU_SPEED]:
    if col not in V_HEADERS:
        raise KeyError(f"Missing V-M field: {col}")
print("Model inputs:", PHONE_CHANNELS)
print("Offline target:", V_ECU_SPEED)

prefix: S=5,642 rows, V=4,884 rows
25% range: S=5,624 rows, V=4,832 rows
50% range: S=5,572 rows, V=7,296 rows
75% range: S=5,592 rows, V=7,354 rows
Transferred 9,437,184 bytes total (bounded ranges only)
Model inputs: ['ACCELEROMETER X (m/s�)', 'ACCELEROMETER Y (m/s�)', 'ACCELEROMETER Z (m/s�)', 'GYROSCOPE Yaw (rad/s)', 'GYROSCOPE Pitch (rad/s)', 'GYROSCOPE Roll (rad/s)']
Offline target: Indicated Vehicle Speed (km/hr)


In [3]:
# 3. Build causal trailing windows and current-speed labels
WINDOW = 26                    # ~2.6 s at the measured 10 Hz cadence
TRAIN_STRIDE = 2               # overlapping windows inside training blocks
EVAL_STRIDE = WINDOW           # non-overlapping evaluation windows
MAX_LABEL_AGE_SECONDS = 0.12
MAX_CONTIGUOUS_GAP_SECONDS = 0.15

def as_float(row, column):
    try:
        return float(row[column])
    except (KeyError, TypeError, ValueError):
        return np.nan

def make_speed_block(block_name, stride):
    s_rows = BLOCKS[block_name]["S"]
    v_rows = BLOCKS[block_name]["V"]

    # Phone timestamps are shifted onto the V-M timebase.
    phone_t = np.asarray([
        as_float(r, S_TIME) / 1000 + PHONE_TO_VM_OFFSET_SECONDS
        for r in s_rows
    ])
    phone_x = np.asarray([
        [as_float(r, c) for c in PHONE_CHANNELS]
        for r in s_rows
    ])
    good_phone = np.isfinite(phone_t) & np.all(np.isfinite(phone_x), axis=1)
    phone_t, phone_x = phone_t[good_phone], phone_x[good_phone]
    order = np.argsort(phone_t)
    phone_t, phone_x = phone_t[order], phone_x[order]
    phone_t, unique_idx = np.unique(phone_t, return_index=True)
    phone_x = phone_x[unique_idx]

    vm_t = np.asarray([as_float(r, V_TIME) for r in v_rows])
    ecu_speed = np.asarray([as_float(r, V_ECU_SPEED) for r in v_rows])
    good_vm = np.isfinite(vm_t) & np.isfinite(ecu_speed)
    vm_t, ecu_speed = vm_t[good_vm], ecu_speed[good_vm]
    if len(vm_t) == 0:
        raise ValueError(f"{block_name}: no valid V-M ECU speed rows")
    order = np.argsort(vm_t)
    vm_t, ecu_speed = vm_t[order], ecu_speed[order]
    vm_t, unique_idx = np.unique(vm_t, return_index=True)
    ecu_speed = ecu_speed[unique_idx]

    # Latest V-M label at or before the phone endpoint: no future label leakage.
    idx = np.searchsorted(vm_t, phone_t, side="right") - 1
    idx_safe = np.clip(idx, 0, len(vm_t) - 1)
    label_age = phone_t - vm_t[idx_safe]
    matched = (
        (idx >= 0) & (label_age >= 0)
        & (label_age <= MAX_LABEL_AGE_SECONDS)
    )
    t = phone_t[matched]
    x = phone_x[matched]
    speed = ecu_speed[idx_safe[matched]]  # km/h; offline training label only

    xs, ys = [], []
    cuts = np.flatnonzero(np.diff(t) > MAX_CONTIGUOUS_GAP_SECONDS) + 1
    runs = np.split(np.arange(len(t)), cuts)
    for run in runs:
        for end in range(WINDOW - 1, len(run), stride):
            row_idx = run[end]
            xs.append(x[run[end - WINDOW + 1:end + 1]])
            ys.append(speed[row_idx])

    X = (np.stack(xs).astype(np.float32) if xs
         else np.empty((0, WINDOW, len(PHONE_CHANNELS)), dtype=np.float32))
    y = np.asarray(ys, dtype=np.float32)
    print(f"{block_name}: windows={len(y):,}, stride={stride}")
    return {"X": X, "y": y}

TRAIN_BLOCKS = ["prefix", "50% range"]
VALIDATION_BLOCK = "25% range"
DIAGNOSTIC_BLOCK = "75% range"  # inspected already; never use for tuning

train_parts = [make_speed_block(name, TRAIN_STRIDE) for name in TRAIN_BLOCKS]
X_train = np.concatenate([d["X"] for d in train_parts])
y_train = np.concatenate([d["y"] for d in train_parts])
eval_data = {
    name: make_speed_block(name, EVAL_STRIDE)
    for name in ["prefix", "50% range", VALIDATION_BLOCK, DIAGNOSTIC_BLOCK]
}
if len(y_train) == 0 or len(eval_data[VALIDATION_BLOCK]["y"]) == 0:
    raise ValueError("No train/validation windows. Check timestamp and column alignment.")

prefix: windows=2,430, stride=2
50% range: windows=2,761, stride=2
prefix: windows=187, stride=26
50% range: windows=214, stride=26
25% range: windows=170, stride=26
75% range: windows=215, stride=26


In [7]:
import copy
import torch.nn.functional as F

# Normalize the target using training data only.
Y_MEAN_V1 = float(y_train.mean())
Y_STD_V1 = max(float(y_train.std()), 1e-6)

y_train_z_v1 = torch.from_numpy(
    ((y_train - Y_MEAN_V1) / Y_STD_V1).astype(np.float32)
)
y_val_z_v1 = torch.from_numpy(
    ((eval_data[VALIDATION_BLOCK]["y"] - Y_MEAN_V1) / Y_STD_V1)
    .astype(np.float32)
)

train_loader_v1 = DataLoader(
    TensorDataset(torch.from_numpy(X_train_norm), y_train_z_v1),
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(7),
)
val_x_v1 = X_val_norm  # already normalized with training-only IMU statistics

class SpeedCNNUncertaintyV1(SpeedCNNUncertainty):
    def forward(self, x):
        z = self.features(x.transpose(1, 2)).squeeze(-1)
        raw = self.head(z)
        mu_z = raw[:, 0]
        var_z = F.softplus(raw[:, 1]) + 1e-4
        return mu_z, var_z

def gaussian_nll_v1(target_z, mu_z, var_z):
    return 0.5 * (
        torch.log(var_z) + (target_z - mu_z).square() / var_z
    ).mean()

# Fresh model and optimizer: train from scratch, not from the saturated v0 weights.
model_v1 = SpeedCNNUncertaintyV1()
optimizer_v1 = torch.optim.AdamW(
    model_v1.parameters(), lr=1e-3, weight_decay=1e-4
)

best_val_nll_v1 = float("inf")
best_state_v1 = None
stale_v1 = 0

for epoch in range(1, 31):
    model_v1.train()
    for xb, yb_z in train_loader_v1:
        optimizer_v1.zero_grad(set_to_none=True)
        mu_z, var_z = model_v1(xb)
        loss = gaussian_nll_v1(yb_z, mu_z, var_z)
        loss.backward()
        nn.utils.clip_grad_norm_(model_v1.parameters(), max_norm=1.0)
        optimizer_v1.step()

    model_v1.eval()
    with torch.no_grad():
        val_mu_z, val_var_z = model_v1(val_x_v1)
        val_nll = gaussian_nll_v1(
            y_val_z_v1, val_mu_z, val_var_z
        ).item()
        val_mu_kmh = Y_MEAN_V1 + Y_STD_V1 * val_mu_z
        val_mae_kmh = torch.mean(
            torch.abs(eval_data[VALIDATION_BLOCK]["y"] - val_mu_kmh.numpy())
        ).item()

    print(
        f"epoch {epoch:02d} | val NLL={val_nll:.3f} "
        f"| val MAE={val_mae_kmh:.2f} km/h"
    )

    if val_nll < best_val_nll_v1:
        best_val_nll_v1 = val_nll
        best_state_v1 = copy.deepcopy(model_v1.state_dict())
        stale_v1 = 0
    else:
        stale_v1 += 1
        if stale_v1 >= 5:
            print("Early stopping on 25% validation NLL.")
            break

model_v1.load_state_dict(best_state_v1)
model_v1.eval()

def report_v1(block_name):
    data = eval_data[block_name]
    x = torch.from_numpy(
        ((data["X"] - X_MEAN) / X_STD).astype(np.float32)
    )
    with torch.no_grad():
        mu_z, var_z = model_v1(x)

    pred_kmh = Y_MEAN_V1 + Y_STD_V1 * mu_z.numpy()
    sigma_kmh = Y_STD_V1 * np.sqrt(var_z.numpy())
    err = pred_kmh - data["y"]
    corr = (
        np.corrcoef(pred_kmh, data["y"])[0, 1]
        if len(data["y"]) > 1 and np.std(pred_kmh) > 0
        and np.std(data["y"]) > 0 else np.nan
    )
    baseline_mae = np.mean(np.abs(Y_MEAN_V1 - data["y"]))
    print(
        f"{block_name}: MAE={np.mean(np.abs(err)):.2f} km/h, "
        f"RMSE={np.sqrt(np.mean(err**2)):.2f}, r={corr:.3f}, "
        f"mean sigma={np.mean(sigma_kmh):.2f} km/h, "
        f"train-mean baseline MAE={baseline_mae:.2f}"
    )

for block in ["prefix", "50% range", "25% range", "75% range"]:
    report_v1(block)  # 75% is diagnostic only; don't tune on it

TypeError: abs(): argument 'input' (position 1) must be Tensor, not numpy.ndarray

In [9]:
import copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Confirm the expected data-preparation variables exist.
for name in ("X_train", "y_train", "eval_data"):
    if name not in globals():
        raise NameError(f"Run the data-preparation cells first; missing {name}")

VALIDATION_BLOCK_V1 = "25% range"
if VALIDATION_BLOCK_V1 not in eval_data:
    raise KeyError(f"Missing validation block: {VALIDATION_BLOCK_V1}")

# Recompute input and target normalization from training data only.
X_train_v1 = np.asarray(X_train, dtype=np.float32)
y_train_v1 = np.asarray(y_train, dtype=np.float32).reshape(-1)

if X_train_v1.ndim != 3 or X_train_v1.shape[-1] != 6:
    raise ValueError(f"Expected X_train shape [N, time, 6], got {X_train_v1.shape}")
if len(X_train_v1) == 0 or len(y_train_v1) == 0:
    raise ValueError("Training set is empty.")

WINDOW_V1 = X_train_v1.shape[1]
X_MEAN_V1 = X_train_v1.mean(axis=(0, 1), keepdims=True).astype(np.float32)
X_STD_V1 = X_train_v1.std(axis=(0, 1), keepdims=True).astype(np.float32)
X_STD_V1 = np.where(X_STD_V1 < 1e-6, 1.0, X_STD_V1).astype(np.float32)

Y_MEAN_V1 = float(y_train_v1.mean())
Y_STD_V1 = max(float(y_train_v1.std()), 1e-6)
y_train_z_v1 = ((y_train_v1 - Y_MEAN_V1) / Y_STD_V1).astype(np.float32)

X_train_norm_v1 = (
    (X_train_v1 - X_MEAN_V1) / X_STD_V1
).astype(np.float32)

val_data_v1 = eval_data[VALIDATION_BLOCK_V1]
X_val_v1 = np.asarray(val_data_v1["X"], dtype=np.float32)
y_val_kmh_np_v1 = np.asarray(val_data_v1["y"], dtype=np.float32).reshape(-1)

if len(X_val_v1) == 0:
    raise ValueError("Validation set is empty.")

X_val_norm_v1 = torch.from_numpy(
    ((X_val_v1 - X_MEAN_V1) / X_STD_V1).astype(np.float32)
)
y_val_z_v1 = torch.from_numpy(
    ((y_val_kmh_np_v1 - Y_MEAN_V1) / Y_STD_V1).astype(np.float32)
)
# Keep the km/h validation labels as a Torch tensor to avoid the previous type error.
y_val_kmh_v1 = torch.from_numpy(y_val_kmh_np_v1)

train_loader_v1 = DataLoader(
    TensorDataset(
        torch.from_numpy(X_train_norm_v1),
        torch.from_numpy(y_train_z_v1),
    ),
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(7),
)

class SpeedCNNUncertaintyV1(nn.Module):
    """Compact CNN; predicts normalized speed mean and positive variance."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(6, 32, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Conv1d(32, 32, kernel_size=5, padding=2),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(32, 48, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(48, 48, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Linear(48, 64),
            nn.GELU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        # x is [batch, time, six IMU channels].
        z = self.features(x.transpose(1, 2)).squeeze(-1)
        raw = self.head(z)
        mu_z = raw[:, 0]
        var_z = F.softplus(raw[:, 1]) + 1e-4
        return mu_z, var_z

def gaussian_nll_v1(target_z, mu_z, var_z):
    return 0.5 * (
        torch.log(var_z) + (target_z - mu_z).square() / var_z
    ).mean()

# Fresh initialization: this does not continue from the failed v1 run.
torch.manual_seed(7)
model_v1 = SpeedCNNUncertaintyV1()  # CPU
optimizer_v1 = torch.optim.AdamW(
    model_v1.parameters(), lr=1e-3, weight_decay=1e-4
)

best_val_nll_v1 = float("inf")
best_state_v1 = None
patience_v1 = 5
stale_epochs_v1 = 0
max_epochs_v1 = 30

for epoch in range(1, max_epochs_v1 + 1):
    model_v1.train()
    train_loss_sum_v1 = 0.0

    for xb, yb_z in train_loader_v1:
        optimizer_v1.zero_grad(set_to_none=True)
        mu_z, var_z = model_v1(xb)
        loss = gaussian_nll_v1(yb_z, mu_z, var_z)
        loss.backward()
        nn.utils.clip_grad_norm_(model_v1.parameters(), max_norm=1.0)
        optimizer_v1.step()
        train_loss_sum_v1 += loss.item() * len(yb_z)

    model_v1.eval()
    with torch.no_grad():
        val_mu_z_v1, val_var_z_v1 = model_v1(X_val_norm_v1)
        val_nll_v1 = gaussian_nll_v1(
            y_val_z_v1, val_mu_z_v1, val_var_z_v1
        ).item()

        # Convert predicted normalized speed back to km/h.
        val_mu_kmh_v1 = Y_MEAN_V1 + Y_STD_V1 * val_mu_z_v1

        # Both operands are Torch tensors — this fixes the reported TypeError.
        val_mae_kmh_v1 = torch.mean(
            torch.abs(y_val_kmh_v1 - val_mu_kmh_v1)
        ).item()

    train_nll_v1 = train_loss_sum_v1 / len(y_train_v1)
    print(
        f"epoch {epoch:02d} | train NLL={train_nll_v1:.3f} "
        f"| val NLL={val_nll_v1:.3f} "
        f"| val MAE={val_mae_kmh_v1:.2f} km/h"
    )

    if val_nll_v1 < best_val_nll_v1:
        best_val_nll_v1 = val_nll_v1
        best_state_v1 = copy.deepcopy(model_v1.state_dict())
        stale_epochs_v1 = 0
    else:
        stale_epochs_v1 += 1
        if stale_epochs_v1 >= patience_v1:
            print("Early stopping on 25% validation NLL.")
            break

if best_state_v1 is None:
    raise RuntimeError("Training did not produce a model state.")

model_v1.load_state_dict(best_state_v1)
model_v1.eval()

TRAIN_MEAN_SPEED_V1 = Y_MEAN_V1

def report_v1(block_name, role):
    data = eval_data[block_name]
    y_np = np.asarray(data["y"], dtype=np.float32).reshape(-1)
    if len(y_np) == 0:
        print(f"{block_name}: no evaluation windows")
        return

    x_np = np.asarray(data["X"], dtype=np.float32)
    x_tensor = torch.from_numpy(
        ((x_np - X_MEAN_V1) / X_STD_V1).astype(np.float32)
    )
    y_tensor = torch.from_numpy(y_np)

    with torch.no_grad():
        mu_z, var_z = model_v1(x_tensor)
        pred_kmh = Y_MEAN_V1 + Y_STD_V1 * mu_z
        sigma_kmh = Y_STD_V1 * torch.sqrt(var_z)

    err = pred_kmh - y_tensor
    pred_np = pred_kmh.numpy()
    sigma_np = sigma_kmh.numpy()

    corr = (
        float(np.corrcoef(pred_np, y_np)[0, 1])
        if len(y_np) > 1 and np.std(pred_np) > 0 and np.std(y_np) > 0
        else float("nan")
    )
    baseline_mae = float(np.mean(np.abs(TRAIN_MEAN_SPEED_V1 - y_np)))

    print(
        f"{block_name} ({role}): n={len(y_np)}, "
        f"MAE={torch.mean(torch.abs(err)).item():.2f} km/h, "
        f"RMSE={torch.sqrt(torch.mean(err.square())).item():.2f} km/h, "
        f"bias={torch.mean(err).item():+.2f} km/h, r={corr:.3f}, "
        f"mean sigma={np.mean(sigma_np):.2f} km/h, "
        f"train-mean baseline MAE={baseline_mae:.2f}"
    )

for block_name in ["prefix", "50% range", "25% range", "75% range"]:
    role = "in-sample" if block_name in ["prefix", "50% range"] else "held-out"
    if block_name == "75% range":
        role = "previously inspected diagnostic only; do not tune on it"
    report_v1(block_name, role)

def predict_current_speed_v1(raw_window):
    """Predict current speed from one raw trailing window in the six-channel order."""
    window = np.asarray(raw_window, dtype=np.float32)
    expected = (WINDOW_V1, 6)
    if window.shape != expected:
        raise ValueError(f"Expected window shape {expected}, got {window.shape}")

    x = torch.from_numpy(
        (((window[None] - X_MEAN_V1) / X_STD_V1).astype(np.float32))
    )
    model_v1.eval()
    with torch.no_grad():
        mu_z, var_z = model_v1(x)

    return {
        "speed_kmh": float((Y_MEAN_V1 + Y_STD_V1 * mu_z).item()),
        "speed_sigma_kmh": float((Y_STD_V1 * torch.sqrt(var_z)).item()),
    }

print("Example prediction:",
      predict_current_speed_v1(eval_data["75% range"]["X"][-1]))

# Save as a new v1 file without overwriting anything.
checkpoint_v1 = Path("speed_cnn_uncertainty_v1.pt")
if checkpoint_v1.exists():
    print("Not overwriting existing checkpoint:", checkpoint_v1.resolve())
else:
    torch.save({
        "state_dict": model_v1.state_dict(),
        "x_mean": X_MEAN_V1,
        "x_std": X_STD_V1,
        "y_mean_kmh": Y_MEAN_V1,
        "y_std_kmh": Y_STD_V1,
        "window_samples": WINDOW_V1,
        "sample_rate_hz": 10,
        "target": "current V-M ECU speed (km/h); training label only",
    }, checkpoint_v1)
    print("Saved:", checkpoint_v1.resolve())

epoch 01 | train NLL=0.418 | val NLL=0.783 | val MAE=14.70 km/h
epoch 02 | train NLL=0.179 | val NLL=0.763 | val MAE=15.05 km/h
epoch 03 | train NLL=-0.027 | val NLL=0.675 | val MAE=13.58 km/h
epoch 04 | train NLL=-0.078 | val NLL=0.630 | val MAE=12.95 km/h
epoch 05 | train NLL=-0.091 | val NLL=0.665 | val MAE=14.13 km/h
epoch 06 | train NLL=-0.087 | val NLL=0.626 | val MAE=13.26 km/h
epoch 07 | train NLL=-0.140 | val NLL=0.608 | val MAE=12.91 km/h
epoch 08 | train NLL=-0.179 | val NLL=0.615 | val MAE=12.84 km/h
epoch 09 | train NLL=-0.222 | val NLL=0.638 | val MAE=12.56 km/h
epoch 10 | train NLL=-0.274 | val NLL=0.741 | val MAE=12.86 km/h
epoch 11 | train NLL=-0.099 | val NLL=1.091 | val MAE=13.76 km/h
epoch 12 | train NLL=-0.233 | val NLL=2.189 | val MAE=14.12 km/h
Early stopping on 25% validation NLL.
prefix (in-sample): n=187, MAE=6.93 km/h, RMSE=10.73 km/h, bias=-1.51 km/h, r=0.765, mean sigma=9.40 km/h, train-mean baseline MAE=14.18
50% range (in-sample): n=214, MAE=7.71 km/h, RM

In [4]:
# 4. Compact 1-D CNN with a speed mean and log-variance head
# Four convolution layers + two dense layers; adapted for 10 Hz phone data.
class SpeedCNNUncertainty(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(6, 32, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Conv1d(32, 32, kernel_size=5, padding=2),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(32, 48, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(48, 48, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Linear(48, 64),
            nn.GELU(),
            nn.Linear(64, 2),  # mean speed (km/h), log variance
        )

    def forward(self, x):
        # Input is [batch, time, six phone IMU channels].
        z = self.features(x.transpose(1, 2)).squeeze(-1)
        raw = self.head(z)
        mu_kmh = raw[:, 0]
        logvar_kmh2 = raw[:, 1].clamp(-8.0, 6.0)
        return mu_kmh, logvar_kmh2

def gaussian_nll_speed(target_kmh, mu_kmh, logvar_kmh2):
    error = target_kmh - mu_kmh
    return 0.5 * (
        logvar_kmh2 + error.square() * torch.exp(-logvar_kmh2)
    ).mean()

model = SpeedCNNUncertainty()
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

Trainable parameters: 21026


In [5]:
# 5. Train on prefix + 50%; early-stop on 25%; inspect 75% only once
# Normalization statistics are fitted on training windows only.
X_MEAN = X_train.mean(axis=(0, 1), keepdims=True).astype(np.float32)
X_STD = X_train.std(axis=(0, 1), keepdims=True).astype(np.float32)
X_STD = np.where(X_STD < 1e-6, 1.0, X_STD).astype(np.float32)
X_train_norm = ((X_train - X_MEAN) / X_STD).astype(np.float32)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train_norm), torch.from_numpy(y_train)),
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(7),
)
val_data = eval_data[VALIDATION_BLOCK]
X_val_norm = torch.from_numpy(
    ((val_data["X"] - X_MEAN) / X_STD).astype(np.float32)
)
y_val = torch.from_numpy(val_data["y"])

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
best_val_nll = float("inf")
best_state = None
patience, stale_epochs = 5, 0
MAX_EPOCHS = 30

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad(set_to_none=True)
        mu, logvar = model(xb)
        loss = gaussian_nll_speed(yb, mu, logvar)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss_sum += loss.item() * len(yb)

    model.eval()
    with torch.no_grad():
        val_mu, val_logvar = model(X_val_norm)
        val_nll = gaussian_nll_speed(y_val, val_mu, val_logvar).item()
        val_mae = torch.mean(torch.abs(y_val - val_mu)).item()

    train_nll = train_loss_sum / len(y_train)
    print(
        f"epoch {epoch:02d} | train NLL={train_nll:.3f} | "
        f"val NLL={val_nll:.3f} | val MAE={val_mae:.2f} km/h"
    )
    if val_nll < best_val_nll:
        best_val_nll = val_nll
        best_state = copy.deepcopy(model.state_dict())
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= patience:
            print("Early stopping on validation NLL.")
            break

if best_state is None:
    raise RuntimeError("Training did not produce a model checkpoint.")
model.load_state_dict(best_state)
model.eval()

TRAIN_MEAN_SPEED = float(np.mean(y_train))

def report_block(name, data, role):
    if len(data["y"]) == 0:
        print(f"{name}: no evaluation windows")
        return
    x = torch.from_numpy(((data["X"] - X_MEAN) / X_STD).astype(np.float32))
    y = data["y"]
    with torch.no_grad():
        mu, logvar = model(x)
        pred = mu.numpy()
        sigma = torch.exp(0.5 * logvar).numpy()
    err = pred - y
    corr = (
        float(np.corrcoef(pred, y)[0, 1])
        if len(y) > 1 and np.std(pred) > 0 and np.std(y) > 0
        else float("nan")
    )
    baseline_mae = float(np.mean(np.abs(TRAIN_MEAN_SPEED - y)))
    print(
        f"{name} ({role}): n={len(y)}, MAE={np.mean(np.abs(err)):.2f} km/h, "
        f"RMSE={np.sqrt(np.mean(err**2)):.2f}, bias={np.mean(err):+.2f}, "
        f"r={corr:.3f}, mean predicted sigma={np.mean(sigma):.2f} km/h, "
        f"train-mean baseline MAE={baseline_mae:.2f}"
    )

for name in ["prefix", "50% range", VALIDATION_BLOCK, DIAGNOSTIC_BLOCK]:
    role = "in-sample" if name in TRAIN_BLOCKS else "held-out"
    if name == DIAGNOSTIC_BLOCK:
        role = "previously inspected diagnostic only; do not tune on it"
    report_block(name, eval_data[name], role)

epoch 01 | train NLL=211.294 | val NLL=3.674 | val MAE=19.52 km/h
epoch 02 | train NLL=3.353 | val NLL=3.414 | val MAE=14.81 km/h
epoch 03 | train NLL=3.252 | val NLL=3.398 | val MAE=14.42 km/h
epoch 04 | train NLL=3.221 | val NLL=3.435 | val MAE=14.70 km/h
epoch 05 | train NLL=3.201 | val NLL=3.379 | val MAE=13.82 km/h
epoch 06 | train NLL=3.182 | val NLL=3.368 | val MAE=13.44 km/h
epoch 07 | train NLL=3.159 | val NLL=3.368 | val MAE=13.29 km/h
epoch 08 | train NLL=3.142 | val NLL=3.435 | val MAE=14.65 km/h
epoch 09 | train NLL=3.126 | val NLL=3.361 | val MAE=13.12 km/h
epoch 10 | train NLL=3.113 | val NLL=3.396 | val MAE=13.98 km/h
epoch 11 | train NLL=3.101 | val NLL=3.425 | val MAE=14.57 km/h
epoch 12 | train NLL=3.122 | val NLL=3.537 | val MAE=16.63 km/h
epoch 13 | train NLL=3.078 | val NLL=3.479 | val MAE=15.68 km/h
epoch 14 | train NLL=3.077 | val NLL=3.399 | val MAE=13.79 km/h
Early stopping on validation NLL.
prefix (in-sample): n=187, MAE=7.51 km/h, RMSE=10.33, bias=-2.25, r=

In [6]:
# 6. Current-window inference helper and safe checkpoint save
# Supply 26 raw phone samples in PHONE_CHANNELS order; no GPS/V-M input at inference.
def predict_current_speed(raw_window):
    window = np.asarray(raw_window, dtype=np.float32)
    if window.shape != (WINDOW, 6):
        raise ValueError(f"Expected {(WINDOW, 6)}, received {window.shape}")
    x = ((window[None] - X_MEAN) / X_STD).astype(np.float32)
    model.eval()
    with torch.no_grad():
        mu, logvar = model(torch.from_numpy(x))
    sigma = torch.exp(0.5 * logvar)
    return {
        "speed_kmh": float(mu.item()),
        "speed_sigma_kmh": float(sigma.item()),
    }

# The checkpoint is saved only if that filename does not already exist.
checkpoint_path = Path("speed_cnn_uncertainty_v0.pt")
if checkpoint_path.exists():
    print(f"Not overwriting existing file: {checkpoint_path.resolve()}")
else:
    torch.save({
        "state_dict": model.state_dict(),
        "x_mean": X_MEAN,
        "x_std": X_STD,
        "phone_channels": PHONE_CHANNELS,
        "window_samples": WINDOW,
        "sample_rate_hz": 10,
        "target": "current V-M ECU speed (km/h), training label only",
    }, checkpoint_path)
    print("Saved:", checkpoint_path.resolve())

# Example using one evaluation window; diagnostic only, not an accuracy claim.
example_window = eval_data["75% range"]["X"][-1]
print("Example prediction:", predict_current_speed(example_window))

Saved: C:\Users\kandp\dead-reckoning-learning\DR_learning\speed_cnn_uncertainty_v0.pt
Example prediction: {'speed_kmh': 21.069061279296875, 'speed_sigma_kmh': 20.08553695678711}


In [10]:
val = eval_data[VALIDATION_BLOCK]
x_val = torch.from_numpy(
    ((val["X"] - X_MEAN_V1) / X_STD_V1).astype(np.float32)
)
y_val_np = np.asarray(val["y"], dtype=np.float32)

model_v1.eval()
with torch.no_grad():
    mu_z, var_z = model_v1(x_val)
    mu_kmh = Y_MEAN_V1 + Y_STD_V1 * mu_z
    sigma_kmh = Y_STD_V1 * torch.sqrt(var_z)

abs_error = np.abs(y_val_np - mu_kmh.numpy())
sigma = sigma_kmh.numpy()

print(f"Validation MAE: {abs_error.mean():.2f} km/h")
print(f"Mean/median sigma: {sigma.mean():.2f} / {np.median(sigma):.2f} km/h")
print(f"Within 1 sigma: {np.mean(abs_error <= sigma):.1%} (Gaussian reference ~68%)")
print(f"Within 1.96 sigma: {np.mean(abs_error <= 1.96 * sigma):.1%} (reference ~95%)")
print(
    "Correlation of sigma with absolute error:",
    np.corrcoef(sigma, abs_error)[0, 1]
    if np.std(sigma) > 0 and np.std(abs_error) > 0 else np.nan
)

Validation MAE: 12.91 km/h
Mean/median sigma: 14.64 / 15.01 km/h
Within 1 sigma: 62.4% (Gaussian reference ~68%)
Within 1.96 sigma: 88.2% (reference ~95%)
Correlation of sigma with absolute error: -0.12794322180382706
